# Agg统计多个值

In [1]:
import pandas as pd

# 读取员工数据
df = pd.read_csv(r"D:\AAAjobbing\actfirst\Numpy&Pandas\data\employees.csv")

# 一次计算多个统计值
# 按department_id分组, 计算salary的最小值,中位数,最大值
df.groupby('department_id')['salary'].agg(['min','median','max'])

,min,median,max
department_id,,,
10.0,4400.0,4400.0,4400.0
20.0,6000.0,9500.0,13000.0
30.0,2500.0,2850.0,11000.0
40.0,6500.0,6500.0,6500.0
50.0,2100.0,3100.0,8200.0
60.0,4200.0,4800.0,9000.0
70.0,10000.0,10000.0,10000.0
80.0,6100.0,8900.0,14000.0
90.0,17000.0,17000.0,24000.0


In [3]:
# 针对不同的聚合字段进行不同的聚合操作
# 按department_id分组，统计job_id的种类数，commission_pct的平均值
df.groupby('department_id').agg({'job_id':'nunique','commission_pct':'mean'}).rename(columns={'job_id':'岗位数','commission_pct':'平均值'})

,岗位数,平均值
department_id,,
10.0,1,NaN
20.0,2,NaN
30.0,2,NaN
40.0,1,NaN
50.0,3,NaN
60.0,1,NaN
70.0,1,NaN
80.0,2,0.225
90.0,2,NaN


In [4]:
#  自定义聚合函数操作
def f(x):
    """统计每个部门员工last_name的首字母"""
    result = set()
    for i in x:
        result.add(i[0])
    return result

df.groupby('department_id')['last_name'].agg(f)


# 注意：函数中的df.groupby("department_id")返回的DataFrameGroupBy可以理解为多个组对应的DataFrame的集合。df.groupby("department_id")["last_name"]返回的SeriesGroupBy可以理解为多个组对应的series集合。所以在使用agg或者后面的transform、filter对数据进行处理的时候，传递给自定义函数的参数是一个Series

department_id
10.0                                                   {W}
20.0                                                {F, H}
30.0                                    {C, B, K, T, H, R}
40.0                                                   {M}
50.0     {M, O, S, D, J, L, N, E, A, K, B, V, R, C, W, ...
60.0                                       {A, L, H, P, E}
70.0                                                   {B}
80.0     {M, O, S, D, J, L, H, E, A, B, K, V, R, C, Z, ...
90.0                                                {D, K}
100.0                                   {C, U, S, P, G, F}
110.0                                               {H, G}
Name: last_name, dtype: object

# transform分组转换

In [6]:
# 将每一组的样本数据减去各组的均值，实现数据标准化
df = pd.read_csv(r"D:\AAAjobbing\actfirst\Numpy&Pandas\data\employees.csv")

# 求每一个部门的平均薪水
df.groupby('department_id')['salary'].mean()
# 1)将每一条的数据减去各组的均值,实现数据标准化
df.groupby('department_id')['salary'].transform(lambda x: x - x.mean())

0      4666.666667
1     -2333.333333
2     -2333.333333
3      3240.000000
4       240.000000
          ...     
102   -3500.000000
103       0.000000
104       0.000000
105    1850.000000
106   -1850.000000
Name: salary, Length: 107, dtype: float64

In [11]:
# 2）按分组使用平均值填充缺失值
import numpy as np
df = pd.read_csv(r"D:\AAAjobbing\actfirst\Numpy&Pandas\data\employees.csv")
na_index = pd.Series(df.index.tolist()).sample(30)  # 随机挑选30条数据
df.loc[na_index, "salary"] = pd.NA  # 将这30条数据的salary设置为缺失值
print(df.groupby("department_id")["salary"].agg(["size", "count"]))  # 查看每组数据总数与非空数据数

def fill_missing(x):
    # 使用平均值填充，如果平均值也为NaN，用0填充
    if np.isnan(x.mean()):
        return 0
    return x.fillna(x.mean())

df["salary"] = df.groupby("department_id")["salary"].transform(fill_missing)
print(df.groupby("department_id")["salary"].agg(["size", "count"]))  # 查看每组数据总数与非空数据数


               size  count
department_id             
10.0              1      0
20.0              2      2
30.0              6      4
40.0              1      1
50.0             45     32
60.0              5      3
70.0              1      1
80.0             34     24
90.0              3      2
100.0             6      5
110.0             2      2
               size  count
department_id             
10.0              1      1
20.0              2      2
30.0              6      6
40.0              1      1
50.0             45     45
60.0              5      5
70.0              1      1
80.0             34     34
90.0              3      3
100.0             6      6
110.0             2      2


In [ ]:
# 分组过滤
# 按照分组的属性丢弃若干数据。
commission_pct_filter = df.groupby("department_id").filter(
    lambda x: x["commission_pct"].notnull().all()
)  # 按department_id分组，过滤掉commission_pct包含空值的分组
print(commission_pct_filter)
